# Iceberg `snapsthot` and `add_files` procedures

In [ ]:
%%configure -f
{   
    "numExecutors": 1, 
     "executorMemory": "14G",
     "executorCores": 4,
    "conf":{
        "spark.dynamicAllocation.enabled": "false",
         "spark.sql.extensions":"org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
         "spark.sql.catalog.spark_catalog":"org.apache.iceberg.spark.SparkSessionCatalog",
         "spark.sql.catalog.spark_catalog.type":"glue",
         "spark.hadoop.hive.metastore.client.factory.class":"com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory"
    }
}

Setup variables

In [ ]:
DATABASE_NAME="mydb2"
S3_BUCKET_NAME="my_datalake"
SOURCE_TABLE_NAME="sales_transactions_1"
TARGET_TABLE_NAME="sales_transactions_iceberg_snapshot"

## 1. Create the source table

In [ ]:
# Create database (if not already created)
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
LOCATION 's3://{S3_BUCKET_NAME}/{DATABASE_NAME}/'
""")

In [ ]:
# Create table with day partition
spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {DATABASE_NAME}.{SOURCE_TABLE_NAME} (
    transaction_id STRING,
    total_amount DECIMAL(10,2),
    transaction_time TIMESTAMP,
    day STRING
)
PARTITIONED BY (day)
STORED AS PARQUET
LOCATION 's3://{S3_BUCKET_NAME}/{DATABASE_NAME}/{SOURCE_TABLE_NAME}/'
""")

## 2. Populate the source table

Here we create 10 daily partitions. Each partition with 100 files.

In [ ]:
from pyspark.sql.types import *
import random
from decimal import Decimal
from datetime import datetime, timedelta

spark.conf.set("hive.exec.dynamic.partition.mode", "nonstrict")

# Function to generate random transactions for a specific day
def generate_transactions_for_day(date_str, num_transactions):
    data = []
    
    # Convert date_str (YYYYMMDD) to datetime
    year = int(date_str[:4])
    month = int(date_str[4:6])
    day = int(date_str[6:])
    base_date = datetime(year, month, day)
    
    for _ in range(num_transactions):
        transaction_id = f"TXN_AR_{random.randint(10000000, 99999999)}"
        # Create Decimal directly from random.uniform without using round function
        total_amount = Decimal(str(random.uniform(10.00, 1000.00))).quantize(Decimal('.01'))
        
        # Random time during the day
        hour = random.randint(0, 23)
        minute = random.randint(0, 59)
        second = random.randint(0, 59)
        
        transaction_time = datetime(year, month, day, hour, minute, second)
        
        data.append((transaction_id, total_amount, transaction_time, date_str))
    
    return data

# Schema for the DataFrame
schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("total_amount", DecimalType(10,2), False),
    StructField("transaction_time", TimestampType(), False),
    StructField("day", StringType(), False)
])

In [ ]:
# Base date to start from (current date)
base_date = datetime(2025, 9, 1)

# Write data for 10 days
for i in range(10):
    current_date = base_date + timedelta(days=i)
    date_str = current_date.strftime("%Y%m%d")
    
    # Generate 1000 transactions for the day
    transactions = generate_transactions_for_day(date_str, 1000)
    
    # Create DataFrame
    df = spark.createDataFrame(transactions, schema)
    
    # Write to table
    df.repartition(100).write.mode("append").insertInto(f"{DATABASE_NAME}.{SOURCE_TABLE_NAME}")
    
    print(f"Loaded {len(transactions)} transactions for day {date_str}")

print("Data generation completed!")

## 3. Create the Iceberg table using the snapshot procedure

In [ ]:
spark.sql(f"""
CALL system.snapshot(
    source_table => '{DATABASE_NAME}.{SOURCE_TABLE_NAME}',
    table => '{DATABASE_NAME}.{TARGET_TABLE_NAME}',
    location => 's3://{S3_BUCKET_NAME}/{DATABASE_NAME}/{TARGET_TABLE_NAME}/'
)
""")

In [ ]:
#Verify table is ok
spark.sql(f"""
SELECT * FROM {DATABASE_NAME}.{TARGET_TABLE_NAME}
""").show(truncate=False)

## 4. Confirm operations which delete table files are not allowed in the Iceberg table

This fail

In [ ]:
from datetime import datetime, timedelta

timenow = str(datetime.now())

expire_snapshot_df=spark.sql(f"""
CALL system.expire_snapshots(table => '{DATABASE_NAME}.{TARGET_TABLE_NAME}', older_than => TIMESTAMP '{timenow}',  retain_last => 1)
"""
)

We'll see in the last section of this notebook how to enable full operations

## 5. Add data in source table into 2 new partitions

In [ ]:
# Base date to start from 
base_date = datetime(2025, 9, 11)

# Write data for 2 days
for i in range(2):
    current_date = base_date + timedelta(days=i)
    date_str = current_date.strftime("%Y%m%d")
    
    # Generate 1000 transactions for the day
    transactions = generate_transactions_for_day(date_str, 1000)
    
    # Create DataFrame
    df = spark.createDataFrame(transactions, schema)
    
    # Write to table
    df.repartition(100).write.mode("append").insertInto(f"{DATABASE_NAME}.{SOURCE_TABLE_NAME}")
    
    print(f"Loaded {len(transactions)} transactions for day {date_str}")

print("Data generation completed!")

In [ ]:
#validate the source table has now the new data
spark.sql(f"""
SELECT COUNT(*) as num_transactions, day FROM {DATABASE_NAME}.{SOURCE_TABLE_NAME} WHERE day = '20250911' OR day = '20250912' GROUP BY day
""").show(truncate=False)

You can see the Iceberg table is not in sync

In [ ]:
spark.sql(f"""
SELECT COUNT(*) as num_transactions, day FROM {DATABASE_NAME}.{TARGET_TABLE_NAME} WHERE day = '20250911' OR day = '20250912' GROUP BY day
""").show(truncate=False)

## 6. Use the `add_files` procedure to add files from the source table 2 new partitions into the Iceberg table

Let's add the first partition using table name as source_table:

In [ ]:
spark.sql(f"""
    CALL system.add_files(
        source_table => '{DATABASE_NAME}.{SOURCE_TABLE_NAME}', 
        table => '{DATABASE_NAME}.{TARGET_TABLE_NAME}',
        partition_filter => map('day', '20250911')
    )
""").show(truncate=False)

In [ ]:
# Verify current Iceberg table state
spark.sql(f"""
SELECT COUNT(*) as num_transactions, day FROM {DATABASE_NAME}.{TARGET_TABLE_NAME} WHERE day = '20250911' OR day = '20250912' GROUP BY day
""").show(truncate=False)

Let's add the first partition using the table path as source_table:

In [ ]:
spark.sql(f"""
    CALL system.add_files(
        source_table => '`parquet`.`s3://{S3_BUCKET_NAME}/{DATABASE_NAME}/{SOURCE_TABLE_NAME}/`', 
        table => '{DATABASE_NAME}.{TARGET_TABLE_NAME}',
        partition_filter => map('day', '20250912')
    )
""").show(truncate=False)

In [ ]:
# Verify current Iceberg table state
spark.sql(f"""
SELECT COUNT(*) as num_transactions, day FROM {DATABASE_NAME}.{TARGET_TABLE_NAME} WHERE day = '20250911' OR day = '20250912' GROUP BY day
""").show(truncate=False)

## 7. Add data in source table into an existing partition
Note this will create an additional file in the `day=20250910` partition

In [ ]:
#Add new data in source table inside an existing partition 
spark.sql(f"""
    INSERT INTO {DATABASE_NAME}.{SOURCE_TABLE_NAME}
    VALUES ('TXN_AR_NEW99999', 299.99, TIMESTAMP '2025-09-10 18:30:00', '20250910')
""")

In [ ]:
#Confirm source table state
spark.sql(f"""
SELECT COUNT(*) as num_transactions, day FROM {DATABASE_NAME}.{SOURCE_TABLE_NAME} WHERE day = '20250910' GROUP BY day
""").show(truncate=False)

You can see the Iceberg table is not in sync

In [ ]:
# Verify current Iceberg table state
spark.sql(f"""
SELECT COUNT(*) as num_transactions, day FROM {DATABASE_NAME}.{TARGET_TABLE_NAME} WHERE day = '20250910' GROUP BY day
""").show(truncate=False)

## 8. Use the `add_files` procedure to add files from the source table existing partition into the Iceberg table
Note this will fail

In [ ]:
spark.sql(f"""
    CALL system.add_files(
        source_table => '{DATABASE_NAME}.{SOURCE_TABLE_NAME}', 
        table => '{DATABASE_NAME}.{TARGET_TABLE_NAME}',
        partition_filter => map('day', '20250910')
    )
""").show(truncate=False)

## 9. To fix the issue, delete the existing partition data and then use the `add_files` procedure to add files from the source table existing partition into the Iceberg table

In [ ]:
spark.sql(f"""
    DELETE FROM {DATABASE_NAME}.{TARGET_TABLE_NAME}
    WHERE day = '20250910'
""")

In [ ]:
spark.sql(f"""
    CALL system.add_files(
        source_table => '{DATABASE_NAME}.{SOURCE_TABLE_NAME}', 
        table => '{DATABASE_NAME}.{TARGET_TABLE_NAME}',
        partition_filter => map('day', '20250910')
    )
""").show(truncate=False)

In [ ]:
# Verify current Iceberg table state
spark.sql(f"""
SELECT COUNT(*) as num_transactions, day FROM {DATABASE_NAME}.{TARGET_TABLE_NAME} WHERE day = '20250910' GROUP BY day
""").show(truncate=False)

## 10. Enable full operations on Iceberg table
We saw in step 4 that operations which delete table files are not allowed in the Iceberg table. We can allow them setting the `gc.enabled` table property to `true`.

**WARNING**: Only change `gc.enabled` when you're ready to fully transition away from the source table. This change allows operations that could corrupt your source table's data files.

In [ ]:
spark.sql(f"""
ALTER TABLE {DATABASE_NAME}.{TARGET_TABLE_NAME}
SET TBLPROPERTIES ('gc.enabled' = 'true')
""")  

Now this works

In [ ]:
from datetime import datetime, timedelta

timenow = str(datetime.now())

expire_snapshot_df=spark.sql(f"""
CALL system.expire_snapshots(table => '{DATABASE_NAME}.{TARGET_TABLE_NAME}', older_than => TIMESTAMP '{timenow}',  retain_last => 1)
"""

**WARNING**: Only change gc.enabled when you're ready to fully transition away from the source table. This change allows operations that could corrupt your source table's data files.